# Install Required Libraries

# Stable Library Versions Due to dependencies issues with import Trainer from transformers

1. TODO: Make sure Python Version 3.11.9 is used
2. TODO: Use virtual python environment (.venv) to keep project installations clean

**Use: `pip uninstall -y torch torchvision torchaudio transformers accelerate` if you have any conflicts and then reinstall according to the below versions. These are the main imports**

In [ ]:
# Stable Library Versions Due to dependencies issues with import Trainer from transformers

# TODO: Make sure Python Version 3.11.9 is used
# TODO: Use virtual python environment (.venv) to keep project installations clean

# Use: 'pip uninstall -y torch torchvision torchaudio transformers accelerate' if you have any conflicts and then reinstall according to the below versions. These are the main imports

# =========================
# STEP 4: Install GPU PyTorch (CUDA 11.8)
# =========================
%pip install -U pip

# 1. Fix core binary compatibility FIRST
%pip install "numpy<2"

# 2. PyTorch CUDA 12.1 (GPU enabled)
%pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu118

# 3. HuggingFace ecosystem (compatible set)
%pip install transformers==4.36.2 accelerate==0.27.2 datasets evaluate scikit-learn rouge_score bert_score

# 4. Optional extras (safe)
%pip install ipywidgets

## Test that CUDA GPU is being detect and used

In [ ]:
import torch
from transformers import Trainer

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

# All the Necessary packages can be Imported in this Section

In [21]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

import torch
import numpy as np
import accelerate
from datasets import load_dataset
import evaluate
import time

from rouge_score import rouge_scorer
from bert_score import score as bert_score
from nltk.translate.bleu_score import corpus_bleu
from nltk.translate.meteor_score import meteor_score
import nltk


# Loading the Dataset, Splitting dataset into Train and Test set

In [ ]:
# TODO: Change the Filepath to the Complete Dataset
file_path = "../../data/dataset.csv"

dataset = load_dataset("csv", data_files=file_path)

# 1. Extract and convert the column values to a standard Python list of integers
binary_values = [1 if val else 0 for val in dataset["train"]["mgt"]]

# 2. Remove the old boolean column
dataset["train"] = dataset["train"].remove_columns("mgt")

# 3. Add the values back under the same column name
dataset["train"] = dataset["train"].add_column("mgt", binary_values)

dataset = dataset["train"].rename_column("mgt", "labels")

split_dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print(train_dataset[:5])
print(test_dataset[:5])

# Intialize Model, Prepare setup for training

1. Choose the Correct Model to Train with. 
    1. Use distilbert to ensure your workflow works
    2. Else use your assigned model

In [ ]:
# Test Setup with Light Model
#model_id = "distilbert-base-uncased"

model_id = "Davlan/afro-xlmr-large" # This is the AfroXLMR-large model

# Requires Permission Access on hugging Face
# Make sure you login into huggingface before using
#model_id = "Jacaranda/Xhosa_ZuluLlama3_v1" # Xhosa_ZuluLlama3 Model


#model_id = "FacebookAI/xlm-roberta-base" # XLM-RoBERTa-base Model


tokenizer = AutoTokenizer.from_pretrained(model_id)

def preprocess(examples):
  return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )

tokenized_train = train_dataset.map(preprocess, batched=True)
tokenized_test = test_dataset.map(preprocess, batched=True)

tokenized_train = tokenized_train.remove_columns(["id", "source", "text"])
tokenized_test  = tokenized_test.remove_columns(["id", "source", "text"])

# Intialize the Specific Model using Model_id. 2 Labels for MGT/Not MGT
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels = 2)

print(f"Total parameters to train: {sum(p.numel() for p in model.parameters()):,}")

Trainging and Evaluation Setups

In [ ]:
# Load all classification metrics
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")


data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy  = accuracy_metric.compute(predictions=predictions, references=labels)
    precision = precision_metric.compute(predictions=predictions, references=labels, average="weighted")
    recall    = recall_metric.compute(predictions=predictions, references=labels, average="weighted")
    f1        = f1_metric.compute(predictions=predictions, references=labels, average="weighted")

    return {
        "accuracy":  accuracy["accuracy"],
        "precision": precision["precision"],
        "recall":    recall["recall"],
        "f1":        f1["f1"],
    }



# Setup Training Arguments
# TrainingArguments for each model were selected based on hardware available to ensure kernels do not crash during training
if model_id == "distilbert-base-uncased":
    print(f"Model Load: {model_id}")
    full_train_args = TrainingArguments(
        output_dir="../../data/full_results/test",
        num_train_epochs=7,
        per_device_train_batch_size=16,       # 277M fits comfortably at 16
        per_device_eval_batch_size=16,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        save_total_limit=2,
        metric_for_best_model="accuracy",
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=10,
        fp16=True,  
        dataloader_num_workers=2,
        dataloader_pin_memory=True,                          
        seed=42,
    )
elif model_id == "FacebookAI/xlm-roberta-base":
    print(f"Model Load: {model_id}")
    full_train_args = TrainingArguments(
        output_dir="../../data/full_results/xlmr_base",
        num_train_epochs=7,
        per_device_train_batch_size=16,       # 277M fits comfortably at 16
        per_device_eval_batch_size=16,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        save_total_limit=2,
        metric_for_best_model="accuracy",
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=10,
        fp16=True,                            # RTX 5060 supports fp16 well
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        seed=42,
    )
elif model_id == "Davlan/afro-xlmr-large":
    print(f"Model Load: {model_id}")
    full_train_args = TrainingArguments(
        output_dir="../../data/full_results/afro_xlmr",
        num_train_epochs=5,                   # large model converges faster
        per_device_train_batch_size=8,        # halved due to larger model size
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=2,        # simulates batch size of 16
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        save_total_limit=2,
        metric_for_best_model="accuracy",
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=10,
        fp16=True,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        seed=42,
    )
elif model_id == "Jacaranda/Xhosa_ZuluLlama3_v1":
    print(f"Model Load: {model_id}")
    full_train_args = TrainingArguments(
        output_dir="../../data/full_results/zululLama3",
        num_train_epochs=3,                   # 8B converges quickly, more risks overfitting
        per_device_train_batch_size=2,        # very small batch — 8B is heavy even on 16GB
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,        # simulates batch size of 16
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        save_total_limit=2,
        metric_for_best_model="accuracy",
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=10,
        fp16=True,
        gradient_checkpointing=True,          # trades compute for memory — essential for 8B
        dataloader_pin_memory=True,
        dataloader_num_workers=2,
        seed=42,
    )
else:
    raise ValueError(f"No TrainingArguments configured for model_id: {model_id}")

full_trainer = Trainer(
    model = model,
    args=full_train_args,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_test,
    data_collator = data_collator,
    compute_metrics = compute_metrics
)

For BLEU, METEOR, ROUGE-L and BERTScore Calculation Setup

In [ ]:
nltk.download("wordnet")

# generated_text - Text that was generated with other LLM's. MGT
# reference_texts = Text that was gathered from datasets. Not MGT (HGT)
def compute_similarity_metrics(generated_texts, reference_texts):
    # BLEU
    references_tokenized  = [[ref.split()] for ref in reference_texts]
    hypotheses_tokenized  = [gen.split() for gen in generated_texts]
    bleu = corpus_bleu(references_tokenized, hypotheses_tokenized)

    # METEOR
    meteor_scores = [
        meteor_score([ref.split()], gen.split())
        for ref, gen in zip(reference_texts, generated_texts)
    ]
    avg_meteor = np.mean(meteor_scores)

    # ROUGE-L
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_scores = [
        scorer.score(ref, gen)["rougeL"].fmeasure
        for ref, gen in zip(reference_texts, generated_texts)
    ]
    avg_rouge = np.mean(rouge_scores)

    # BERTScore
    P, R, F1 = bert_score(generated_texts, reference_texts, lang="en")
    avg_bertscore = F1.mean().item()

    return {
        "BLEU":      bleu,
        "METEOR":    avg_meteor,
        "ROUGE-L":   avg_rouge,
        "BERTScore": avg_bertscore,
    }

Train and Evaluate the Model and Calculate Metrics

In [ ]:
# Timer to Track Duration
start_time = time.time()

# Train Model
full_trainer.train()

# Evaluate classification metrics
full_eval_results = full_trainer.evaluate()
full_duration = time.time() - start_time
print(f"Full Fine-tuning took: {full_duration:.2f} seconds")
print(f"Classification Results: {full_eval_results}")

# Get predictions on test set
predictions_output = full_trainer.predict(tokenized_test)
logits = predictions_output.predictions
predicted_labels = np.argmax(logits, axis=-1)
print(f"Predicted Labels Distribution: {np.bincount(predicted_labels)}")


# Separate texts by class for similarity comparison
human_texts = [text for text, label in zip(test_dataset["text"], test_dataset["labels"]) if label == 0]
mgt_texts   = [text for text, label in zip(test_dataset["text"], test_dataset["labels"]) if label == 1]

# Compute similarity metrics (MGT vs Human text)
similarity_results = compute_similarity_metrics(mgt_texts, human_texts)
print(f"Similarity Results: {similarity_results}")